In [ ]:
# Reload and re-export of the Palantir-integrated object.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl

In [ ]:
# Save result
adata.write('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
adata.layers['scaled'] = adata.X

In [ ]:
print(adata.X)

In [ ]:
  del adata.X

In [ ]:
adata

In [ ]:
adata.write('/home/yanjun2/scanpy/8_figures/forrds.h5ad')

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/8_figures/figures_new"
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='condition')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_condition')

In [ ]:
sc.pl.umap(adata, color = 'leiden_res_0_8')

In [ ]:
c5 = adata[adata.obs['leiden_res_0_8'] == 'Mac5']

In [ ]:
sc.tl.leiden(
    c5,
    resolution= 0.3,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_0_8'
)

In [ ]:
sc.pl.umap(c5, color = ['leiden_res_0_8', 'LYVE1'], size = 50)

In [ ]:
c5.obs['leiden_res_0_8'] = c5.obs['leiden_res_0_8'].map(
    {
        '0' : 'Mac5',
        '1' : 'Mac5',
        '2' : 'Mac3'
    }
)

In [ ]:
rest = adata[~adata.obs['leiden_res_0_8'].isin(['Mac5', 'Mac6'])]

In [ ]:
sc.tl.embedding_density(c5, basis='umap', groupby='condition')
sc.pl.embedding_density(c5, basis='umap', key='umap_density_condition')

In [ ]:
c6 = adata[adata.obs['leiden_res_0_8'] == 'Mac6']

In [ ]:
sc.tl.leiden(
    c6,
    resolution= 0.3,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_0_8'
)

In [ ]:
sc.tl.embedding_density(c6, basis='umap', groupby='leiden_res_0_8')
sc.pl.embedding_density(c6, basis='umap', key='umap_density_leiden_res_0_8')

In [ ]:
sc.pl.embedding_density(c6, basis='umap', key='umap_density_condition')

In [ ]:
sc.pl.umap(c6, color = ['leiden_res_0_8', 'LYVE1'], size = 50)

In [ ]:
c6.obs['leiden_res_0_8'] = c6.obs['leiden_res_0_8'].map(
    {
        '0' : 'Mac6',
        '1' : 'Mac4',
        '2' : 'Mac6'
    }
)

In [ ]:
sc.pl.umap(adata, color = 'leiden_res_0_8', size = 50)

In [ ]:
c5_indices = c5.obs[c5.obs['leiden_res_0_8'] == 'Mac3'].index
c6_indices = c6.obs[c6.obs['leiden_res_0_8'] == 'Mac4'].index
c5_indices

In [ ]:
# Identify indices of interesting cells in c5 and c6
c5_indices = c5.obs[c5.obs['leiden_res_0_8'] == 'Mac3'].index
c6_indices = c6.obs[c6.obs['leiden_res_0_8'] == 'Mac4'].index

# Update the original adata.obs DataFrame to reflect the new groupings
adata.obs.loc[c5_indices, 'leiden_res_0_8'] = 'Mac3'
adata.obs.loc[c6_indices, 'leiden_res_0_8'] = 'Mac4'

In [ ]:
sc.pl.umap(adata, color = 'leiden_res_0_8', size = 50)

In [ ]:
mdi = adata[adata.obs['condition'] == 'MDI']
mdix = adata[adata.obs['condition'] == 'MDIX']

In [ ]:
obs_df = mdix.obs[['leiden_res_0_8', 'timepoint']]
frequency_df = obs_df.groupby(['timepoint', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]
fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  
for cluster, container in zip(percentage_df.columns, ax.containers):
    if cluster in ['Mac3', 'Mac4']:  # Adjust if cluster names are numerical or categorical
        for bar in container:
            if bar.get_width() > 0:  # Only annotate non-zero bars
                width = bar.get_width()
                ax.text(
                    bar.get_x() + width / 2,  # Center text horizontally in the bar
                    bar.get_y() + bar.get_height() / 2,  # Center text vertically in the bar
                    f'{width:.1f}',  # Format text to 1 decimal place
                    ha='center', va='center', fontsize=5, color='white'
                )


plt.title("Proportion of clusters in MDIX treatment")
plt.xlabel("Percentage (%)")
plt.ylabel("Timepoint")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
fig.savefig("/home/yanjun2/scanpy/8_figures/figures_new/mdix_freq_plot_rotated.pdf", dpi=600)
plt.show()


In [ ]:
obs_df = mdi.obs[['leiden_res_0_8', 'timepoint']]
frequency_df = obs_df.groupby(['timepoint', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]
fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  
for cluster, container in zip(percentage_df.columns, ax.containers):
    if cluster in ['Mac3', 'Mac4']:  # Adjust if cluster names are numerical or categorical
        for bar in container:
            if bar.get_width() > 0:  # Only annotate non-zero bars
                width = bar.get_width()
                ax.text(
                    bar.get_x() + width / 2,  # Center text horizontally in the bar
                    bar.get_y() + bar.get_height() / 2,  # Center text vertically in the bar
                    f'{width:.1f}',  # Format text to 1 decimal place
                    ha='center', va='center', fontsize=5, color='white'
                )


plt.title("Proportion of clusters in MDI treatment")
plt.xlabel("Percentage (%)")
plt.ylabel("Timepoint")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
fig.savefig("/home/yanjun2/scanpy/8_figures/figures_new/mdi_freq_plot_rotated.pdf", dpi=600)
plt.show()


In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", method="wilcoxon", layer = 'data')
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/home/yanjun2/scanpy/8_figures/figures_new/deg.csv', index=False)

In [ ]:
adata

In [ ]:
cell = ['CTGCCATAGTATGAAC-1_3']
umap = pd.DataFrame(adata.obsm['X_umap'][:,0:2], index=adata.obs_names, columns=['x', 'y'])
plt.scatter(umap["x"], umap["y"], s=5, color="lightgrey")
plt.scatter(umap.loc[cell, "x"], umap.loc[cell, "y"], s=30)